In [2]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [3]:
load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [4]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/09/15/ai-in-productio

In [ ]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

#this is just a system prompt nothing else using one shot pronmpting which has an example

In [6]:
#just joins our user prompt with all the links in the website

def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [7]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer

In [8]:
# here is where we ask an llm to select relevant and absolute links rather than relative links 
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [9]:
select_relevant_links("https://www.vinaychanamallu.com/")

{'links': [{'type': 'LinkedIn profile',
   'url': 'https://www.linkedin.com/in/vinay-chanamallu/'},
  {'type': 'GitHub profile', 'url': 'https://github.com/VinChan2001'},
  {'type': 'GitHub repository',
   'url': 'https://github.com/VinChan2001/catt-crash-clearance'},
  {'type': 'GitHub repository',
   'url': 'https://github.com/VinChan2001/medical_image_classification'},
  {'type': 'GitHub repository',
   'url': 'https://github.com/VinChan2001/Predicting-High-Booking-Rate-Airbnb-Listings'},
  {'type': 'GitHub repository',
   'url': 'https://github.com/VinChan2001/Flight-Delay-Prediction-Pipeline'},
  {'type': 'GitHub repository',
   'url': 'https://github.com/VinChan2001/customer_churn_prediction'},
  {'type': 'GitHub repository',
   'url': 'https://github.com/VinChan2001/RAG_Chatbot'},
  {'type': 'GitHub repository',
   'url': 'https://github.com/VinChan2001/financial-market-ml-analysis'},
  {'type': 'GitHub repository',
   'url': 'https://github.com/VinChan2001/Time-series-forecasti

In [10]:
# fetch_website_contents -> gets all the content (text) info from the website
# and we fetch the relevant links 
# now we are stitching it all together like 
# first landing page content and relevant links one by one 
# then write the link and again get the contents in the link (cheeky!)
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [11]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
openbmb/MiniCPM5-1B
Updated
5 days ago
•
28.8k
•
603
nvidia/LocateAnything-3B
Updated
4 days ago
•
18.3k
•
488
meituan-longcat/LongCat-Video-Avatar-1.5
Updated
5 days ago
•
409
HauhauCS/Qwen3.6-35B-A3B-Uncensored-HauhauCS-Aggressive
Updated
Apr 17
•
2.23M
•
1.1k
bytedance-research/Lance
Updated
3 days ago
•
2.86k
•
980
Browse 2M+ models
Spaces
Running
on
Zero
Agents
Featured
169
LongCat-Vi

In [21]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

In [22]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [15]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nopenbmb/MiniCPM5-1B\nUpdated\n5 days ago\n•\n28.8k\n•\n604\nnvidia/LocateAnything-3B\nUpdated\n4 days ago\n•\n18.3k\n•\n488\nH

In [23]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [17]:
create_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is a pioneering AI community and platform dedicated to building the future of machine learning. It serves as a collaborative hub where data scientists, researchers, and developers come together to create, discover, and share cutting-edge models, datasets, and AI applications.

The platform hosts over **2 million AI models** and **500,000+ datasets**, empowering users to accelerate innovation in natural language processing (NLP), computer vision, audio analysis, reinforcement learning, and more.

---

## What We Offer

- **Models**: Access, share, and collaborate on millions of pre-trained models spanning a wide range of ML tasks. Stay ahead with trending state-of-the-art models updated regularly.
  
- **Datasets**: Discover and contribute to an extensive repository of diverse datasets curated for AI training and research purposes.
  
- **Spaces**: Build and deploy AI apps and demos directly on the platform, showcasing live interactive models.
  
- **Buckets & Storage**: Secure and scalable cloud storage solutions tailored for machine learning workflows.
  
- **HuggingChat**: Cutting-edge AI chat applications powered by community-driven models.

- **Enterprise Solutions**: Dedicated team and enterprise support, including specialized tools such as inference endpoints, providers, and Hugging Face PRO for advanced enterprise use cases.

---

## Our Community & Culture

Hugging Face is built around an open-source and collaborative ethos. The community includes researchers, engineers, and enthusiasts passionate about advancing AI responsibly and ethically. Community engagement is fostered through:

- A vibrant **Discord server** and **discussion forums**
- Active contributions on **GitHub**
- Regularly updated **blog posts** and tutorials on AI research and development
- Hosting AI challenges, leaderboards, and sharing daily research papers

The culture encourages continuous learning, innovation, and open collaboration, providing a welcoming environment for newcomers and industry veterans alike.

---

## Customers & Use Cases

Hugging Face serves a broad spectrum of users:

- **Researchers & Academics**: Sharing and accessing models and datasets that catalyze new AI research.
  
- **Startups & Developers**: Quickly prototyping and deploying ML-powered applications using ready-to-use models and Spaces.
  
- **Large Enterprises**: Leveraging enterprise-grade support and infrastructure to integrate AI at scale, with tailored solutions for inference and model deployment.
  
- **Community Contributors**: Individual contributors who develop models, datasets, or AI tools and share them globally.

Use cases include natural language understanding, image and video generation, audio-driven applications, and robotics, among others.

---

## Careers at Hugging Face

Join a forward-thinking team dedicated to advancing AI for everyone! Hugging Face looks for passionate engineers, researchers, product managers, and community leaders who thrive in a collaborative, open-source environment.

- Work alongside leading experts in machine learning and AI
- Contribute to open-source projects shaping the future of AI technology
- Engage with a supportive community that values innovation, diversity, and ethical AI development

Visit [Hugging Face Careers](https://huggingface.co/careers) to explore current job opportunities and become a part of this vibrant AI movement.

---

## Connect with Us

- Explore and collaborate: [huggingface.co](https://huggingface.co)
- Join our Discord community for discussions and support
- Follow our GitHub for the latest open-source projects
- Read insightful blogs and daily papers on AI advancements

Together, let's build the future of machine learning!

---

Hugging Face — **The AI community building the future.**

In [18]:
create_brochure("Vinay's Portfolio", "https://www.vinaychanamallu.com/")

# Vinay's Portfolio Brochure

---

## About Vinay Chanamallu  
Vinay Chanamallu is a seasoned Data Scientist and Machine Learning Engineer with 3 years of professional experience and over 30 projects completed. Based in Maryland, USA, Vinay recently completed a Master of Science in Business Analytics from the University of Maryland (expected May 2026) and holds an AWS ML Specialty certification. He is open to relocation and available for new opportunities.

Vinay specializes in building production-ready machine learning systems focused on payments risk, churn prediction, health scoring, and A/B experimentation leveraging cloud platforms such as AWS and GCP. His work transforms complex data into intelligent, actionable solutions across industries.

---

## Core Competencies & Skills  

### Machine Learning & Engineering  
- Deployment and optimization of ML models in production environments  
- Expertise in XGBoost, LightGBM, CatBoost, and deep learning (PyTorch, TensorFlow, Transformers)  
- Proficient in statistical modeling and causal inference  

### Cloud Platforms  
- AWS: SageMaker, Lambda, Glue, Redshift  
- GCP: Vertex AI, Cloud Run, Pub/Sub, Dataflow  

### Data Tools & Technologies  
- Programming: Python, SQL, R, Pandas, NumPy  
- MLOps: Docker, Kubernetes, MLflow, Airflow, Git  
- Big Data: Spark, Databricks, Kafka  
- Visualization: Tableau, Plotly, Matplotlib, Seaborn, D3.js  

---

## Professional Experience  

- **Data Engineer (ML / GenAI Focus)** | heal.ID (Oct 2025 - Present)  
- **Data Scientist Intern** | EPAM Systems (Feb 2022 - Jul 2024)  
- **Research Assistant** | Jawaharlal Nehru Technological University (Aug 2021 - Feb 2022)  

Vinay has successfully deployed over 5 machine learning models supporting high-impact areas such as payments risk assessment, customer churn analysis, and health scoring through rigorous A/B testing and experimentation frameworks.

---

## Projects & Impact  

Vinay’s portfolio includes 24 projects spanning machine learning, AI, data analytics, and dashboard development. Notable work includes a featured project in transportation safety analytics—CATT Lab Crash Clearance—which demonstrates his ability to apply data science to real-world challenges.

---

## Company Culture & Values  

While Vinay’s portfolio is individual-focused, the work ethos emphasizes:  
- **Innovation:** Leveraging cutting-edge cloud and ML technologies  
- **Impact:** Delivering solutions that drive tangible business improvements  
- **Continuous Learning:** Keeping current with emerging technologies and methodologies  
- **Collaboration:** Working across teams and disciplines to deploy robust solutions efficiently  

---

## Opportunities & Careers  

Vinay is actively seeking new roles in data science and machine learning engineering that allow him to apply his expertise in ML model development and deployment across cloud platforms. He welcomes collaborations and employment offers in environments that value data-driven decision making and continuous experimentation.

---

## Contact  

- Location: Maryland, USA (Willing to relocate)  
- Available for opportunities in Data Science, ML Engineering, AI, and Analytics  
- Portfolio & Resume Links: [Discover My Work →](#)  

---

Leverage Vinay’s expertise to develop scalable, production ML systems that turn complex data into intelligent business solutions. Whether you are an organization seeking data-driven innovation or a team looking to enrich with cutting-edge ML talent, Vinay’s proven skillset is ready to drive your projects forward.

In [24]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [25]:
stream_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face: Where AI Gets a Hug 🤗

---

## Welcome to Hugging Face!

Not just a cute name—Hugging Face is *the* thriving AI community shaping the future of machine learning. Think of it as the ultimate playground for data wizards, model magicians, and AI dreamers all over the globe.

### What’s the Big Deal?

- **2 MILLION+ models** hosted and growing!
- **Hundreds of thousands of datasets** ready for your next big breakthrough.
- **Spaces & Buckets** – Your personal cloud stage for hosting, sharing, and experimenting with AI apps.
- **HuggingChat** – Chatting with AI just got friendlier (and maybe even funnier).

Hugging Face is not just a platform; it’s a revolution in openness, collaboration, and ethical AI-building. Your AI toolbox just got a serious upgrade.

---

## For The Machine Learning Jedi (That’s You!)

- Browse models to find everything from text wizards to video magicians.
- Collaborate and share datasets like an open-source party.
- Explore cutting-edge apps and Spaces where creativity runs wild.
- Follow trending projects (like **LongCat-Video-Avatar** 🐾 and **LocateAnything** 🕵️‍♂️).

All in one place where innovation meets community spirit—because AI works better when we work together.

---

## Culture: Hugging Face Is The Friendliest AI Lab Around

- **Open community:** Whether you’re a newbie or a hardcore AI scientist, you’ll find welcoming folks in Discord, the Forum, and GitHub.
- **Ethics first:** Building AI that is open, responsible, and benefits all humanity. No robot overlords here.
- **Fun meets focus:** Deep research and daily papers coexist with wacky projects and lively chats.
- **Innovate out loud:** Researchers and engineers share their breakthroughs, so we all level up—together.
- **Brand vibe:** Bright yellow (#FFD21E) and orange (#FF9D00) bring energy and warmth, just like their mascot.

---

## Who Uses Hugging Face?

- **Researchers** developing the next-gen AI algorithms.
- **Enterprises** powering smart products with their PRO support and inference endpoints.
- **Startups and hobbyists** creating AI applications without breaking the bank.
- **Educators and learners** diving into datasets and models to fuel knowledge.

If AI is your playground, Hugging Face is your clubhouse.

---

## Join the Hugging Face Family

Looking for a place that values your curiosity and talent? Hugging Face seeks:

- Machine Learning Engineers and Data Scientists eager to push boundaries.
- Researchers ready to explore ethical AI innovations.
- Devs who want to build community-powered tools.
- AI enthusiasts who want their work to actually *make a difference*.

Grow your career while hugging the future of AI!

---

## Quick Hug Recap:

| Hugging Face Perk                  | Why You'll Love It                     |
|-----------------------------------|--------------------------------------|
| 2M+ Models & 500k+ Datasets       | Endless AI inspiration & resources   |
| Collaborative platform + community | Share, learn, build together         |
| Enterprise PRO & Support options  | Ready for serious AI deployments     |
| Open-source spirit + ethics focus | AI for the good of all humanity      |
| Vibrant culture & brand personality | Because AI should be fun and friendly|

---

## Stay Connected

- Dive into the [Hugging Face Blog](https://huggingface.co/blog) for the latest AI stories and tutorials.
- Join the community on [Discord](https://discord.gg/huggingface), the [Forum](https://discuss.huggingface.co), or [GitHub](https://github.com/huggingface).
- Explore the vast [Model Hub](https://huggingface.co/models) and start hugging AI today.

---

Hugging Face — Where AI gets a little friendlier, a lot more collaborative, and the future looks very smart indeed.

# Ready to get hugged? 🤗 Dive in at [huggingface.co](https://huggingface.co) today!